# 02 · Visualise Results & Benchmark Analysis

This notebook loads results logged in `results/runs/*.csv` and `results/epochs/*.csv` and generates publication-quality figures:
- **Summary Table**: Test accuracy (mean ± std across seeds) per model and data fraction
- **Label Efficiency Curves**: Test accuracy vs training set size on log scale
- **Learning Curves**: Training & validation loss/accuracy across epochs
- **Confusion Matrix**: Class-level breakdown of model predictions
- **Runtime & Compute Profiling**: Time per epoch and total compute per member
- **Reproducibility Check**: Verification of identical configuration hashes
- **Cross-Sensor Transfer**: EuroSAT-trained checkpoints evaluated zero-shot on UC Merced (`results/transfer/*.csv`)
- **Few-Shot Adaptation**: linear probes on growing UC Merced subsets, unmodified pretrained backbones vs. backbones fine-tuned on EuroSAT (`init_from`)

Figures are automatically exported to `figures/` as both PNG and PDF.

In [ ]:
# Environment Setup & Imports
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator

# Find repository root (whether running from repo root or notebooks/ folder)
REPO_DIR = Path.cwd()
while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / "pyproject.toml").exists():
    REPO_DIR = REPO_DIR.parent

if not (REPO_DIR / "pyproject.toml").exists():
    raise RuntimeError(f"Could not find repository root with pyproject.toml from {Path.cwd()}")

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.chdir(REPO_DIR)

%load_ext autoreload
%autoreload 2

import satbench
from satbench import load_runs, load_epochs

RESULTS_DIR = REPO_DIR / "results"
FIG_DIR = REPO_DIR / "figures"
SAVE_FIGS = True

print(f"satbench {satbench.__version__} | Repository: {REPO_DIR}")

## 1. Load Experiment Results

Reads all completed runs and epoch metrics logged in `results/`.
Runs produced with uncommitted code changes (`git_dirty == True`) are dropped: their `config_hash` does not identify the code that produced them, so they can silently disagree with clean runs of the same hash.

In [ ]:
INCLUDE_DIRTY = True  # Set to True to keep runs made with uncommitted code changes

runs = load_runs(RESULTS_DIR)
if not runs.empty and not INCLUDE_DIRTY:
    dirty = runs["git_dirty"].astype(str).str.lower().eq("true")
    if dirty.any():
        print(f"Dropped {dirty.sum()} runs with git_dirty=True: "
              + ", ".join(f"{u}={n}" for u, n in runs[dirty].groupby("user").size().items()))
    runs = runs[~dirty].reset_index(drop=True)
epochs = load_epochs(RESULTS_DIR, runs)  # only epochs of the kept runs

if runs.empty:
    print("No completed runs found in results/ yet!")
    print("To generate results:")
    print("1. Run 'notebooks/01_run_experiments.ipynb'")
    print("2. Or run CLI in terminal: uv run satbench run --config configs/smoke_test.yaml --user demo")
else:
    print(f"Loaded {len(runs)} completed runs and {len(epochs)} epoch logs.")
    summary_df = (runs.groupby(["experiment", "user", "platform", "device_name"])
                      .agg(runs=("run_id", "size"),
                           compute_hours=("total_time_s", lambda s: round(s.sum() / 3600, 2)),
                           last_finished=("finished_at", "max")))
    display(summary_df)

## 2. Select Experiment & Plotting Styles

Choose an experiment to visualize. By default, it selects the latest non-smoke-test experiment.

Results from different `pipeline_version`s are never mixed: by default each model uses only its newest pipeline version (so e.g. re-running just the SSL4EO models under a new version replaces their old runs without touching the other models). Set `PIPELINE_VERSION` to pin one version for all models.

In [ ]:
if not runs.empty:
    EXPERIMENT = None        # None = auto-select most recent experiment (prioritizing non-smoke-test)
    INCLUDE_DEBUG = False    # Set to True to include runs with eval_limit (e.g. smoke tests)
    PIPELINE_VERSION = None  # None = newest pipeline_version per model; or e.g. 2 to pin one version
    BASE_LR = 1e-4           # All analyses use runs at this lr; linear probes at other lrs are overlaid
    LR_SWEEP_EXPERIMENTS = ["linear_probe_lr"]  # extra experiments holding linear probes at other lrs
    EXTRA_EXPERIMENTS = ["vit_label_efficiency"]  # extra models added to the label-efficiency plot only
    SIDE_EXPERIMENTS = ["ucm_fewshot_linear_probe"]  # analysed in their own sections (10); never auto-selected here

    if EXPERIMENT is None:
        cand = runs[~runs["experiment"].isin(["smoke_test", *LR_SWEEP_EXPERIMENTS, *EXTRA_EXPERIMENTS, *SIDE_EXPERIMENTS])]
        EXPERIMENT = (cand if not cand.empty else runs).sort_values("finished_at")["experiment"].iloc[-1]

    df = runs[runs["experiment"] == EXPERIMENT].copy()
    if not INCLUDE_DEBUG:
        df_clean = df[df["eval_limit"].isna()]
        if not df_clean.empty:
            df = df_clean
        else:
            print(f"Note: Only debug/subsampled runs found for '{EXPERIMENT}'. Showing them.")

    # Never mix pipeline versions within a model
    df["pipeline_version"] = pd.to_numeric(df["pipeline_version"])
    if PIPELINE_VERSION is None:
        df = df[df["pipeline_version"] == df.groupby("model")["pipeline_version"].transform("max")]
    else:
        df = df[df["pipeline_version"] == int(PIPELINE_VERSION)]
    pipeline_versions = df.groupby("model")["pipeline_version"].first().to_dict()

    # Linear probes at lr != BASE_LR (from this experiment or the lr sweep, same pipeline version)
    # go into df_lr and are only overlaid on the label-efficiency plot; df keeps BASE_LR runs only
    sweep = runs[runs["experiment"].isin(LR_SWEEP_EXPERIMENTS) & runs["model"].isin(pipeline_versions)].copy()
    if not INCLUDE_DEBUG:
        sweep = sweep[sweep["eval_limit"].isna()]
    sweep["pipeline_version"] = pd.to_numeric(sweep["pipeline_version"])
    sweep = sweep[sweep["pipeline_version"] == sweep["model"].map(pipeline_versions)]
    df_lr = pd.concat([df, sweep])
    df_lr = df_lr[df_lr["model"].str.endswith("_linear_probe") & ~np.isclose(df_lr["lr"], BASE_LR)]
    df = df[np.isclose(df["lr"], BASE_LR)]

    # Models from EXTRA_EXPERIMENTS (e.g. ViT runs) at BASE_LR, newest pipeline version per model;
    # kept in df_extra so the other analyses still only cover EXPERIMENT
    df_extra = runs[runs["experiment"].isin(EXTRA_EXPERIMENTS) & ~runs["model"].isin(pipeline_versions)].copy()
    if not INCLUDE_DEBUG:
        df_extra = df_extra[df_extra["eval_limit"].isna()]
    df_extra["pipeline_version"] = pd.to_numeric(df_extra["pipeline_version"])
    df_extra = df_extra[df_extra["pipeline_version"] == df_extra.groupby("model")["pipeline_version"].transform("max")]
    df_extra = df_extra[np.isclose(df_extra["lr"], BASE_LR)]

        # Model styling dictionary
    MODEL_STYLE = {
        # Baseline: Raw pixel classifier
        "linear": dict(
            label="Linear (raw pixels)",
            color="#7f7f7f",
            marker="^",
            ls=":",
        ),
        # Random initialization
        "resnet50_random": dict(
            label="ResNet-50, Random (fine-tune)",
            color="#d62728",
            marker="s",
            ls="-",
        ),
        "resnet50_random_linear_probe": dict(
            label="ResNet-50, Random (linear probe)",
            color="#d62728",
            marker="s",
            ls="--",
        ),
        # ImageNet-1k pretraining
        "resnet50_imagenet": dict(
            label="ResNet-50, ImageNet (fine-tune)",
            color="#1f77b4",
            marker="o",
            ls="-",
        ),
        "resnet50_imagenet_linear_probe": dict(
            label="ResNet-50, ImageNet (linear probe)",
            color="#1f77b4",
            marker="o",
            ls="--",
        ),
        # SSL4EO-S12 domain pretraining
        "resnet50_ssl4eo": dict(
            label="ResNet-50, SSL4EO (fine-tune)",
            color="#2ca02c",
            marker="D",
            ls="-",
        ),
        "resnet50_ssl4eo_linear_probe": dict(
            label="ResNet-50, SSL4EO (linear probe)",
            color="#2ca02c",
            marker="D",
            ls="--",
        ),
        # ViT-B/16 (from EXTRA_EXPERIMENTS)
        "vit_b16_random": dict(
            label="ViT-B/16, Random (fine-tune)",
            color="#ff7f0e",
            marker="v",
            ls="-",
        ),
        "vit_b16_random_linear_probe": dict(
            label="ViT-B/16, Random (linear probe)",
            color="#ff7f0e",
            marker="v",
            ls="--",
        ),
        "vit_b16_imagenet": dict(
            label="ViT-B/16, ImageNet (fine-tune)",
            color="#9467bd",
            marker="P",
            ls="-",
        ),
        "vit_b16_imagenet_linear_probe": dict(
            label="ViT-B/16, ImageNet (linear probe)",
            color="#9467bd",
            marker="P",
            ls="--",
        ),
    }


    def style(model):
        if model not in MODEL_STYLE:
            MODEL_STYLE[model] = dict(
                label=model,
                color=plt.cm.tab10(len(MODEL_STYLE) % 10),
                marker="D",
                ls="-."
            )
        return MODEL_STYLE[model]

    models = [m for m in MODEL_STYLE if m in set(df["model"])] + sorted(set(df["model"]) - set(MODEL_STYLE))
    extra_models = [m for m in MODEL_STYLE if m in set(df_extra["model"])] + sorted(set(df_extra["model"]) - set(MODEL_STYLE))

    def savefig(fig, name):
        if SAVE_FIGS:
            FIG_DIR.mkdir(parents=True, exist_ok=True)
            for ext in ("png", "pdf"):
                fig.savefig(FIG_DIR / f"{EXPERIMENT}__{name}.{ext}", bbox_inches="tight", dpi=200)

    print(f"Experiment : {EXPERIMENT}")
    print(f"Total runs : {len(df)} at lr={BASE_LR:g}")
    print(f"Models     : {models}")
    print(f"Pipeline   : " + ", ".join(f"{m}=v{pipeline_versions[m]}" for m in models))
    print(f"Seeds      : {sorted(int(s) for s in df['seed'].unique())}")
    print(f"Users      : {sorted(df['user'].unique())}")
    print(f"Extra      : {len(df_extra)} runs from {EXTRA_EXPERIMENTS}: {extra_models}")
    print(f"Other lrs  : {len(df_lr)} linear-probe runs at lr="
          + (", ".join(f"{lr:g}" for lr in sorted(df_lr["lr"].unique())) or "-"))
else:
    df = df_lr = df_extra = pd.DataFrame()
    models = extra_models = []

## 3. Summary Table: Test Accuracy (Mean ± Std over Seeds)

Computes the mean test accuracy and standard deviation across seeds for each training fraction.

In [ ]:
if not df.empty:
    per_seed = (df.groupby(["model", "train_fraction", "seed"], as_index=False)
                  .agg(n_train=("n_train", "first"),
                       test_acc=("test_acc", "mean"),
                       test_macro_f1=("test_macro_f1", "mean"),
                       replicates=("run_id", "size")))

    summary = (per_seed.groupby(["model", "train_fraction"], as_index=False)
                 .agg(n_train=("n_train", "first"),
                      n_seeds=("seed", "nunique"),
                      acc_mean=("test_acc", "mean"),
                      acc_std=("test_acc", "std"),
                      f1_mean=("test_macro_f1", "mean")))
    summary["acc_std"] = summary["acc_std"].fillna(0.0)

    cell = (summary["acc_mean"].mul(100).map("{:.1f}".format) + " ± "
            + summary["acc_std"].mul(100).map("{:.1f}".format)
            + " (n=" + summary["n_seeds"].astype(str) + ")")

    table = summary.assign(cell=cell).pivot(index=["train_fraction", "n_train"], columns="model", values="cell")
    display(table[[m for m in models if m in table.columns]])
else:
    print("No experiment runs to display.")

## 4. Label Efficiency Curve

Test accuracy (%) as a function of the number of training samples (log scale). Shaded bands show ± 1 standard deviation across seeds.

In [ ]:
# Choose epoch count to display (e.g. 10, 25, 50), or set None to auto-select max available
EPOCHS_TO_PLOT = 25
INCLUDE_EXTRA = True  # add models from EXTRA_EXPERIMENTS (e.g. ViT) to the plot


def summarise_label_efficiency(d, keys=("model",)):
    """Average replicates within a seed, then mean/std across seeds per (keys, train_fraction)."""
    keys = list(keys)
    per_seed = (d.groupby(keys + ["train_fraction", "seed"], as_index=False)
                 .agg(n_train=("n_train", "first"),
                      test_acc=("test_acc", "mean"),
                      test_macro_f1=("test_macro_f1", "mean"),
                      replicates=("run_id", "size")))
    summary = (per_seed.groupby(keys + ["train_fraction"], as_index=False)
                 .agg(n_train=("n_train", "first"),
                      n_seeds=("seed", "nunique"),
                      acc_mean=("test_acc", "mean"),
                      acc_std=("test_acc", "std"),
                      f1_mean=("test_macro_f1", "mean")))
    summary["acc_std"] = summary["acc_std"].fillna(0.0)
    return per_seed, summary


def format_label_efficiency_axes(ax, summary, title):
    ticks = summary.drop_duplicates("train_fraction").sort_values("n_train")
    ax.set_xscale("symlog", linthresh=300)
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xticks(ticks["n_train"])
    ax.set_xticklabels([f"{n}\n({100 * f:g}%)" for n, f in zip(ticks["n_train"], ticks["train_fraction"])])
    ax.set_xlabel("Labelled training images (fraction of train split)", fontsize=10)
    ax.set_ylabel("Test accuracy (%)", fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=True, facecolor="white", framealpha=0.9)


if not df.empty:
    avail_epochs = sorted(int(e) for e in df["epochs"].dropna().unique()) if "epochs" in df.columns else []
    selected_epochs = EPOCHS_TO_PLOT if EPOCHS_TO_PLOT is not None else (avail_epochs[-1] if avail_epochs else None)
    epoch_label = f" ({selected_epochs} epochs)" if selected_epochs else ""

    # 1. Filter by epoch count to avoid mixing runs; optionally add the extra experiments' models
    plot_models = models + (extra_models if INCLUDE_EXTRA else [])
    src = pd.concat([df, df_extra]) if INCLUDE_EXTRA else df
    df_plot = src[src["epochs"] == selected_epochs] if (selected_epochs is not None and "epochs" in src.columns) else src.copy()

    if df_plot.empty:
        print(f"No runs found with epochs = {selected_epochs}. Available epochs: {avail_epochs}")
    else:
        # 2. Recompute per-seed and summary specifically for this epoch count
        per_seed_plot, summary_plot = summarise_label_efficiency(df_plot)

        # 3. Plot label efficiency (all models at BASE_LR)
        fig, ax = plt.subplots(figsize=(16, 10))
        for m in plot_models:
            st, s = style(m), summary_plot[summary_plot["model"] == m].sort_values("n_train")
            if s.empty:
                continue
            ax.plot(s["n_train"], 100 * s["acc_mean"], marker=st["marker"], ls=st["ls"],
                    color=st["color"], label=st["label"], lw=1.8, ms=6)
            #ax.fill_between(s["n_train"], 100 * (s["acc_mean"] - s["acc_std"]),
            #                100 * (s["acc_mean"] + s["acc_std"]), color=st["color"], alpha=0.15, lw=0)
            p = per_seed_plot[per_seed_plot["model"] == m]
            #ax.scatter(p["n_train"], 100 * p["test_acc"], color=st["color"], s=16, alpha=0.4)

        extra_label = f" + {', '.join(EXTRA_EXPERIMENTS)}" if INCLUDE_EXTRA and extra_models else ""
        format_label_efficiency_axes(ax, summary_plot,
                                     f"Label Efficiency · {EXPERIMENT}{extra_label}{epoch_label} · lr={BASE_LR:g}")
        fig.tight_layout()

        # Save with epoch suffix so different runs do not overwrite each other
        fig_name = f"label_efficiency_{selected_epochs}ep" if selected_epochs else "label_efficiency"
        savefig(fig, fig_name)
        plt.show()


### 4b. Linear Probes: Learning-Rate Comparison

Linear probes at `BASE_LR` (filled markers, solid) against the same probes trained at other learning rates (hollow markers, one line style per lr). Only models with runs at another lr are shown.

In [ ]:
if not df.empty and not df_lr.empty:
    lr_runs = df_lr[df_lr["epochs"] == selected_epochs] if selected_epochs is not None else df_lr
    lr_models = [m for m in MODEL_STYLE if m in set(lr_runs["model"])]
    base_runs = df[df["model"].isin(lr_models)]
    base_runs = base_runs[base_runs["epochs"] == selected_epochs] if selected_epochs is not None else base_runs

    if lr_runs.empty:
        print(f"No linear probes at other lrs with epochs = {selected_epochs}.")
    else:
        _, lr_summary = summarise_label_efficiency(pd.concat([base_runs, lr_runs]), keys=("model", "lr"))
        lr_summary = lr_summary[lr_summary["train_fraction"] > 0]  # drop the untrained 0-label point
        all_lrs = sorted(lr_summary["lr"].unique())
        shown_lrs = [all_lrs[0], all_lrs[-1]] if len(all_lrs) > 1 else all_lrs  # smallest and largest lr only
        other_lrs = [lr for lr in shown_lrs if not np.isclose(lr, BASE_LR)]
        LR_LS = [":", "-.", (0, (5, 1)), (0, (1, 1))]

        fig, ax = plt.subplots(figsize=(12, 8))
        for m in lr_models:
            st = style(m)
            for lr in shown_lrs:
                s = lr_summary[(lr_summary["model"] == m) & np.isclose(lr_summary["lr"], lr)].sort_values("n_train")
                if s.empty:
                    continue
                is_base = np.isclose(lr, BASE_LR)
                ax.plot(s["n_train"], 100 * s["acc_mean"], marker=st["marker"],
                        mfc=st["color"] if is_base else "white",
                        ls="-" if is_base else LR_LS[other_lrs.index(lr) % len(LR_LS)],
                        color=st["color"], label=f"{st['label']}, lr={lr:g}", lw=1.8, ms=6)
                #ax.fill_between(s["n_train"], 100 * (s["acc_mean"] - s["acc_std"]),
                #                100 * (s["acc_mean"] + s["acc_std"]), color=st["color"],
                #                alpha=0.15 if is_base else 0.08, lw=0)

        format_label_efficiency_axes(ax, lr_summary, "")
        ax.set_title("Linear probe (frozen backbone): learning-rate comparison", fontsize=17)  # same style as the 4c panels
        fig.tight_layout()
        fig_name = f"linear_probe_lr_{selected_epochs}ep" if selected_epochs else "linear_probe_lr"
        savefig(fig, fig_name)
        plt.show()
else:
    print("No linear probes at other learning rates to compare.")


### 4c. Presentation Figure: Label Efficiency

Slide-ready version of the label-efficiency plot. Fine-tuning and linear probing get their own panel; colour encodes the pretraining and line style the architecture. The untrained 0% point is replaced by a chance line, and a single legend below the panels explains colours and line styles.
Set `BUILD_UP = True` to also save one figure per stage (raw pixels, + Random, + ImageNet, + SSL4EO, + ViT) so you can step through them on consecutive slides; the axes stay fixed across stages.


In [ ]:
SHOW_BANDS = False  # faint ± std bands across seeds (only drawn where a model has > 1 seed)
BUILD_UP = True     # also save one figure per build-up stage

PRETRAIN_STYLE = {"random": ("Random", "#d62728"), "imagenet": ("ImageNet", "#1f77b4"),
                  "ssl4eo": ("SSL4EO", "#2ca02c")}
ARCH_STYLE = {"resnet50": dict(name="ResNet-50", short="ResNet", ls="-", marker="o"),
              "vit_b16": dict(name="ViT-B/16", short="ViT", ls="--", marker="^")}
STAGES = [("Raw pixels", ["linear"]),
          ("+ Random init", ["resnet50_random"]),
          ("+ ImageNet", ["resnet50_imagenet"]),
          ("+ SSL4EO", ["resnet50_ssl4eo"]),
          ("+ ViT-B/16", ["vit_b16_random", "vit_b16_imagenet"])]


from matplotlib.patches import Patch


def pres_style(model):
    """Colour from the pretraining, line style/marker from the architecture; grey dotted for raw pixels."""
    if model == "linear":
        return dict(label="Raw pixels", color="#7f7f7f", ls=":", marker="s")
    arch, pretrain = model.removesuffix("_linear_probe").rsplit("_", 1)
    name, color = PRETRAIN_STYLE[pretrain]
    return dict(label=f"{name} {ARCH_STYLE[arch]['short']}", color=color, **{k: ARCH_STYLE[arch][k] for k in ("ls", "marker")})


if not df.empty:
    pres_src = pd.concat([df, df_extra])
    pres_src = pres_src[(pres_src["epochs"] == selected_epochs) & (pres_src["train_fraction"] > 0)]
    _, pres = summarise_label_efficiency(pres_src)
    pres_models = [m for m in models + extra_models if m in set(pres["model"])]

    confusions = df["test_confusion_json"].dropna()
    n_classes = len(json.loads(confusions.iloc[0])) if not confusions.empty else 10
    chance = 100 / n_classes
    ticks = pres.drop_duplicates("train_fraction").sort_values("n_train")
    seeds = (pres[pres["model"] != "linear"]
             .assign(arch=lambda d: d["model"].str.removesuffix("_linear_probe").str.rsplit("_", n=1).str[0])
             .groupby("arch")["n_seeds"].max())
    footnote = (f"EuroSAT test accuracy, mean over seeds, lr={BASE_LR:g}, {selected_epochs} epochs")

    def draw_pres(visible):
        panels = [("Fine-tuning", [m for m in pres_models if not m.endswith("_linear_probe")]),
                  ("Linear probe (frozen backbone)", [m for m in pres_models if m.endswith("_linear_probe") or m == "linear"])]
        with plt.rc_context({"font.size": 14, "axes.titlesize": 17, "axes.labelsize": 15,
                             "xtick.labelsize": 13, "ytick.labelsize": 13}):
            fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
            for ax, (title, panel_models) in zip(axes, panels):
                for m in panel_models:
                    if not any(m.removesuffix("_linear_probe") == v for v in visible):
                        continue
                    st, s = pres_style(m), pres[pres["model"] == m].sort_values("n_train")
                    ax.plot(s["n_train"], 100 * s["acc_mean"], color=st["color"], ls=st["ls"],
                            marker=st["marker"], lw=1.2, ms=4.5)
                    if SHOW_BANDS and s["n_seeds"].max() > 1:
                        ax.fill_between(s["n_train"], 100 * (s["acc_mean"] - s["acc_std"]),
                                        100 * (s["acc_mean"] + s["acc_std"]), color=st["color"], alpha=0.08, lw=0)

                ax.axhline(chance, color="black", ls=(0, (4, 3)), lw=1.2, alpha=0.6)
                ax.text(ticks["n_train"].iloc[0], chance + 1.2, f"chance ({chance:g}%)", fontsize=12, alpha=0.7)
                ax.set_xscale("log")
                ax.xaxis.set_minor_locator(NullLocator())
                ax.set_xticks(ticks["n_train"])
                ax.set_xticklabels([f"{100 * f:g}%\n{n:,}" for n, f in zip(ticks["n_train"], ticks["train_fraction"])])
                ax.set_xlim(ticks["n_train"].iloc[0] / 1.25, ticks["n_train"].iloc[-1] * 1.25)
                ax.set_ylim(0, 106)
                ax.set_title(title)
                ax.set_xlabel("Labelled training images")
                ax.grid(True, axis="y", alpha=0.3)
                ax.spines[["top", "right"]].set_visible(False)
            axes[0].set_ylabel("Test accuracy (%)")

            # Horizontal legend below the panels: colours (pretraining) first, then line styles (architecture)
            shown_pretrain = [k for k in PRETRAIN_STYLE if any(v.endswith(k) for v in visible)]
            shown_archs = [a for a in ARCH_STYLE if any(v.startswith(a) for v in visible)]
            handles = [Patch(color=PRETRAIN_STYLE[k][1], label=PRETRAIN_STYLE[k][0]) for k in shown_pretrain]
            if "linear" in visible:
                raw = pres_style("linear")
                handles.append(plt.Line2D([], [], color=raw["color"], ls=raw["ls"], marker=raw["marker"],
                                          lw=1.2, ms=4.5, label=raw["label"]))
            handles += [plt.Line2D([], [], color="#444444", ls=ARCH_STYLE[a]["ls"], marker=ARCH_STYLE[a]["marker"],
                                   lw=1.2, ms=4.5, label=ARCH_STYLE[a]["name"]) for a in shown_archs]
            fig.tight_layout(rect=(0, 0.06, 1, 1))
            fig.legend(handles=handles, loc="lower center", ncol=len(handles), bbox_to_anchor=(0.5, 0.0),
                       frameon=False, handlelength=2.5, columnspacing=2.0)
            fig.text(0.5, -0.03, footnote, ha="center", fontsize=11, color="#555555")
        return fig

    if BUILD_UP:
        visible = []
        for i, (_, stage_models) in enumerate(STAGES, start=1):
            visible += stage_models
            fig = draw_pres(visible)
            savefig(fig, f"pres_label_efficiency_{selected_epochs}ep_stage{i}")
            plt.close(fig)

    fig = draw_pres([m for _, ms in STAGES for m in ms])
    savefig(fig, f"pres_label_efficiency_{selected_epochs}ep")
    plt.show()


## 5. Learning Curves per Training Subset Size

Trajectories across training epochs for each subset size. Set `CURVE_METRIC` to `val_acc`, `train_loss`, `val_loss`, `train_acc`, or `val_macro_f1`.

In [ ]:
if not df.empty and not epochs.empty:
    CURVE_METRIC = "val_loss"  # Options: "val_acc", "val_loss", "train_loss", "train_acc", "val_macro_f1"
    EPOCHS_TO_PLOT = 25       # Options: 10, 25, 50, or None to auto-select the max available

    # 1. Resolve available epoch counts and filter runs
    avail_epochs = sorted(int(e) for e in df["epochs"].dropna().unique()) if "epochs" in df.columns else []
    selected_epochs = EPOCHS_TO_PLOT if EPOCHS_TO_PLOT is not None else (avail_epochs[-1] if avail_epochs else None)

    df_plot = df[df["epochs"] == selected_epochs] if (selected_epochs is not None and "epochs" in df.columns) else df.copy()

    # 2. Filter epoch trajectories for only the selected runs
    ep = epochs[epochs["run_id"].isin(df_plot["run_id"])].copy()
    fractions = sorted(f for f in ep["train_fraction"].unique() if f > 0)

    if not fractions:
        print(f"No epoch metrics found for {selected_epochs} epochs. Available epochs: {avail_epochs}")
    else:
        ncols = min(4, len(fractions))
        nrows = int(np.ceil(len(fractions) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(8.4 * ncols, 6.4 * nrows), sharey=True, squeeze=False)
        scale = 100 if CURVE_METRIC.endswith(("acc", "f1")) else 1

        for ax, f in zip(axes.flat, fractions):
            for m in models:
                st = style(m)
                e = (ep[(ep["train_fraction"] == f) & (ep["model"] == m)]
                       .groupby("epoch")[CURVE_METRIC].agg(["mean", "std"]).fillna(0).reset_index())
                if e.empty:
                    continue
                ax.plot(e["epoch"], scale * e["mean"], color=st["color"], ls=st["ls"],
                        marker=st["marker"], ms=3, label=st["label"])
                #ax.fill_between(e["epoch"], scale * (e["mean"] - e["std"]), scale * (e["mean"] + e["std"]),
                #                color=st["color"], alpha=0.15, lw=0)
            ax.set_title(f"{100 * f:g}% of train split")
            ax.set_xlabel("Epoch")
            ax.grid(True, alpha=0.3)

        for ax in axes.flat[len(fractions):]:
            ax.axis("off")

        axes[0, 0].set_ylabel(CURVE_METRIC + (" (%)" if scale == 100 else ""))
        handles, labels = axes[0, 0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc="lower center", ncol=len(models), bbox_to_anchor=(0.5, -0.04))

        epoch_label = f" ({selected_epochs} epochs)" if selected_epochs else ""
        fig.suptitle(f"{CURVE_METRIC} by Epoch · {EXPERIMENT}{epoch_label}", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0.05, 1, 1))

        # Save figure with epoch count to avoid overwriting different run lengths
        fig_name = f"curves_{CURVE_METRIC}_{selected_epochs}ep" if selected_epochs else f"curves_{CURVE_METRIC}"
        savefig(fig, fig_name)
        plt.show()


## 6. Confusion Matrix

Row-normalized confusion matrix (actual vs predicted classes). Shows how errors are distributed among classes.

In [ ]:
if not df.empty and "test_confusion_json" in df.columns:
    available_models = [m for m in models if m in df["model"].values]
    # Set to a specific model/fraction, or keep as None for auto-selection
    CM_MODEL = "resnet50_random"      # e.g. "resnet50_imagenet", "linear"
    CM_FRACTION = 0.01   # e.g. 0.05, 1.0

    if CM_MODEL is None:
        CM_MODEL = "resnet50_imagenet" if "resnet50_imagenet" in available_models else (available_models[0] if available_models else None)
    if CM_FRACTION is None:
        CM_FRACTION = df[df["train_fraction"] > 0]["train_fraction"].min() if not df[df["train_fraction"] > 0].empty else 0.0

    sel = df[(df["model"] == CM_MODEL) & (df["train_fraction"] == CM_FRACTION)]
    valid_cms = [json.loads(s) for s in sel["test_confusion_json"].dropna() if s]

    if valid_cms:
        cm = np.mean([np.array(x) for x in valid_cms], axis=0)
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_norm = np.divide(cm, row_sums, where=row_sums != 0)

        # Class names from splits CSV or EuroSAT dataset defaults
        dataset_name = df["dataset"].iloc[0]
        split_csv = REPO_DIR / "splits" / f"{dataset_name}_split.csv"
        if split_csv.exists():
            names = (pd.read_csv(split_csv).drop_duplicates("label").sort_values("label")["class_name"].tolist())
        else:
            names = [str(i) for i in range(len(cm))]

        fig, ax = plt.subplots(figsize=(7, 6))
        im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
        for i in range(len(cm)):
            for j in range(len(cm)):
                ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=7.5,
                        color="white" if cm_norm[i, j] > 0.5 else "black")
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=45, ha="right")
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names)
        ax.set(xlabel="Predicted", ylabel="True",
               title=f"Confusion Matrix: {style(CM_MODEL)['label']} · {100 * CM_FRACTION:g}% data (n={len(sel)} runs)")
        fig.colorbar(im, ax=ax, fraction=0.046)
        fig.tight_layout()
        savefig(fig, f"confusion_{CM_MODEL}_{CM_FRACTION:g}")
        plt.show()

## 7. Runtime & Compute Profiling

Hardware execution speed (median seconds per epoch) and total compute hours consumed per member.

In [ ]:
if not df.empty and "sec_per_epoch" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(24, 8.4))

    # Time per epoch
    for (m, dev), g in df.groupby(["model", "device_name"]):
        st = style(m)
        g_agg = g.groupby("n_train")["sec_per_epoch"].median().reset_index()
        axes[0].plot(g_agg["n_train"], g_agg["sec_per_epoch"], marker=st["marker"], ls=st["ls"],
                     color=st["color"], label=f"{st['label']} · {dev}")
    axes[0].set(xscale="log", yscale="log", xlabel="Labelled training images",
                ylabel="Seconds per epoch (train + val)", title="Time per epoch (median)")
    axes[0].grid(True, alpha=0.3, which="both")
    axes[0].legend(fontsize=7.5)

    # Compute per member
    cost = df.groupby(["user", "model"])["total_time_s"].sum().unstack(fill_value=0) / 3600
    cost = cost[[m for m in models if m in cost.columns]]
    if not cost.empty:
        cost.plot.bar(stacked=True, ax=axes[1], color=[style(m)["color"] for m in cost.columns], rot=0)
        axes[1].set(ylabel="Compute (hours)", xlabel="", title="Total compute per member")
        axes[1].legend([style(m)["label"] for m in cost.columns], fontsize=8)
    fig.tight_layout()
    savefig(fig, "runtime")
    plt.show()

### 7b. Total Compute Across All Runs

Every completed run in `results/runs/`, regardless of experiment, pipeline version or `git_dirty` (deduplicated by `run_id`, since the per-member CSVs repeat runs). Transfer evaluations are not timed and are not included.

In [ ]:
all_runs = load_runs(RESULTS_DIR)  # unfiltered: all experiments, pipeline versions, dirty commits
if not all_runs.empty:
    all_runs["who"] = all_runs["user"] + " · " + all_runs["device_name"]
    hours = all_runs.pivot_table(index="who", columns="experiment", values="total_time_s",
                                 aggfunc="sum", fill_value=0) / 3600
    hours = hours.loc[hours.sum(axis=1).sort_values().index, hours.sum().sort_values(ascending=False).index]
    total_h = hours.values.sum()

    fig, ax = plt.subplots(figsize=(12, 0.6 * len(hours) + 2))
    hours.plot.barh(stacked=True, ax=ax, colormap="tab10", width=0.7)
    for i, h in enumerate(hours.sum(axis=1)):
        ax.text(h + 0.01 * total_h, i, f"{h:.1f} h", va="center", fontsize=9)
    ax.set(xlabel="Compute (hours, train + val + test)", ylabel="",
           title=f"Total compute: {total_h:.1f} h ({total_h / 24:.1f} days) over {len(all_runs)} runs")
    ax.grid(True, axis="x", alpha=0.3)
    ax.legend(title="Experiment", fontsize=9, loc="lower right")
    fig.tight_layout()
    savefig(fig, "runtime_total")
    plt.show()
    print(f"Total runtime of all {len(all_runs)} runs: {total_h:.2f} h = {total_h / 24:.2f} days")

## 8. Reproducibility Check

Compares runs sharing identical `config_hash`.
Identical hardware should yield zero variance in test accuracy (`acc_range_pp == 0`).

In [ ]:
if not runs.empty and "config_hash" in runs.columns:
    rep = (runs.groupby("config_hash")
               .agg(experiment=("experiment", "first"), model=("model", "first"),
                    train_fraction=("train_fraction", "first"), seed=("seed", "first"),
                    n_runs=("run_id", "size"),
                    users=("user", lambda s: ", ".join(sorted(set(s)))),
                    devices=("device_name", lambda s: ", ".join(sorted(set(s)))),
                    acc_min=("test_acc", "min"), acc_max=("test_acc", "max")))
    rep = rep[rep["n_runs"] > 1].assign(acc_range_pp=lambda d: 100 * (d["acc_max"] - d["acc_min"]))
    if not rep.empty:
        display(rep.sort_values("acc_range_pp", ascending=False))
    else:
        print("No duplicate runs yet (each config has run exactly once).")

## 9. Cross-Sensor Transfer: EuroSAT → UC Merced

Every checkpoint of the label-efficiency sweeps was evaluated **zero-shot** (no further training, no adaptation) on UC Merced, with its 21 classes mapped onto the EuroSAT label space (`UC_MERCED_TO_EUROSAT` in `data.py`). Only 12 UC Merced classes have a EuroSAT counterpart, giving 1 200 images over **7 of the 10 EuroSAT classes**; `Pasture`, `PermanentCrop` and `SeaLake` have no target images, so any prediction of those classes is an error by construction.

The two datasets differ in far more than the label names:

| | EuroSAT (source) | UC Merced (target) |
|---|---|---|
| Sensor | Sentinel-2 satellite | USGS aerial orthophoto |
| Ground sampling distance | 10 m / pixel | 0.3 m / pixel |
| Patch size | 64 × 64 px ≈ **640 m** across | 256 × 256 px ≈ **77 m** across |
| Residential | whole suburbs as texture | a handful of individual houses |

Both are resized to 224 × 224 before entering the network, so the models see a ≈ 30× difference in spatial scale. Reference points on the target: chance is 10 %, the **majority class** (`Residential`, 400 / 1 200) is 33.3 %.

Note on metrics: `transfer_macro_f1` in the CSV averages over all 10 classes, so the three absent classes pull it down by a factor of 0.7. Below it is recomputed over the 7 classes that are actually present. The fraction of predictions that land on an absent class is reported as a direct read-out of how far off-distribution the inputs look to each model.

In [ ]:
from satbench.results import load_transfers

TRANSFER_TARGET = "uc_merced_rgb"
TRANSFER_EPOCHS = selected_epochs if "selected_epochs" in globals() else None

# Class names (same source as the confusion-matrix section)
_split_csv = REPO_DIR / "splits" / "eurosat_rgb_split.csv"
CLASS_NAMES = (pd.read_csv(_split_csv).drop_duplicates("label").sort_values("label")["class_name"].tolist()
               if _split_csv.exists() else
               ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
                "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"])


def transfer_metrics(cm_json):
    """Accuracy, macro-F1 over present classes, absent-class prediction share and per-class recall."""
    cm = np.array(json.loads(cm_json), dtype=float)
    support = cm.sum(1)
    present = support > 0
    tp = cm.diagonal()
    fp, fn = cm.sum(0) - tp, support - tp
    denom = 2 * tp + fp + fn
    f1 = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    return pd.Series({
        "acc": tp.sum() / cm.sum(),
        "macro_f1_present": f1[present].mean(),
        "absent_pred_share": cm[:, ~present].sum() / cm.sum(),
        "recall": np.where(present, tp / np.maximum(support, 1), np.nan),
        "support": support,
    })


tr = load_transfers(RESULTS_DIR)
tr = tr[(tr["status"] == "completed") & (tr["target_dataset"] == TRANSFER_TARGET)
        & (tr["experiment"] != "smoke_test")].copy()

if tr.empty:
    print(f"No transfer results for target '{TRANSFER_TARGET}' in {RESULTS_DIR / 'transfer'}.")
    tr_summary = pd.DataFrame()
else:
    # Attach training hyper-parameters of the source run (lr, pipeline version, epochs, n_train) via config_hash
    run_meta = (runs.drop_duplicates("config_hash")
                    .set_index("config_hash")[["lr", "pipeline_version", "epochs", "n_train"]])
    tr = tr.drop(columns=[c for c in run_meta.columns if c in tr.columns]).join(run_meta, on="config_hash")
    tr["pipeline_version"] = pd.to_numeric(tr["pipeline_version"])
    unmatched = tr["lr"].isna().sum()
    if unmatched:
        print(f"Dropped {unmatched} transfer rows whose source run is not in results/runs.")
        tr = tr[tr["lr"].notna()]

    # Same lr and epoch budget as the rest of the notebook. Pipeline versions are *not* separated here:
    # the v1 -> v2 bump only changed the SSL4EO input scaling (commit 4d62aac), and all SSL4EO checkpoints
    # are v2 anyway. Set TRANSFER_NEWEST_PIPELINE_ONLY = True to apply the strict per-model rule instead.
    TRANSFER_NEWEST_PIPELINE_ONLY = False
    tr = tr[np.isclose(tr["lr"], BASE_LR)]
    if TRANSFER_NEWEST_PIPELINE_ONLY:
        tr = tr[tr["pipeline_version"] == tr.groupby("model")["pipeline_version"].transform("max")]
    else:
        mixed = tr.groupby("model")["pipeline_version"].nunique()
        mixed = mixed[mixed > 1]
        if not mixed.empty:
            print(f"Note: mixing pipeline versions for {', '.join(mixed.index)} (v1 vs v2 differ only in SSL4EO preprocessing).")
    if TRANSFER_EPOCHS is not None and (tr["epochs"] == TRANSFER_EPOCHS).any():
        tr = tr[tr["epochs"] == TRANSFER_EPOCHS]

    tr = pd.concat([tr.reset_index(drop=True), tr["transfer_confusion_json"].apply(transfer_metrics).reset_index(drop=True)], axis=1)
    tr_models = [m for m in MODEL_STYLE if m in set(tr["model"])] + sorted(set(tr["model"]) - set(MODEL_STYLE))
    tr_support = tr["support"].iloc[0]
    tr_present = tr_support > 0
    CHANCE = 100 / len(tr_support)
    MAJORITY = 100 * tr_support.max() / tr_support.sum()

    # Mean / std over seeds per (model, fraction)
    tr_summary = (tr.groupby(["model", "train_fraction"], as_index=False)
                    .agg(n_train=("n_train", "first"), n_seeds=("seed", "nunique"),
                         source_acc=("source_test_acc", "mean"),
                         acc_mean=("acc", "mean"), acc_std=("acc", "std"),
                         f1_present=("macro_f1_present", "mean"),
                         absent_share=("absent_pred_share", "mean")))
    tr_summary["acc_std"] = tr_summary["acc_std"].fillna(0.0)

    print(f"{len(tr)} transfer evals -> {TRANSFER_TARGET} | models: {len(tr_models)} | "
          f"seeds: {sorted(int(s) for s in tr['seed'].dropna().unique())} | epochs: {sorted(tr['epochs'].unique())}")
    print(f"Target: n={int(tr_support.sum())}, present classes = {[c for c, p in zip(CLASS_NAMES, tr_present) if p]}")
    print(f"Chance = {CHANCE:.1f}%  |  majority class ({CLASS_NAMES[int(tr_support.argmax())]}) = {MAJORITY:.1f}%")

    # Table: full-data checkpoints and the best fraction per model
    full = tr_summary[tr_summary["train_fraction"] == tr_summary["train_fraction"].max()].set_index("model")
    trained = tr_summary[tr_summary["train_fraction"] > 0]  # the untrained 0% head is not a candidate
    best = trained.loc[trained.groupby("model")["acc_mean"].idxmax()].set_index("model")
    table = pd.DataFrame({
        "EuroSAT acc (100%)": 100 * full["source_acc"],
        "UCM acc (100%)": 100 * full["acc_mean"],
        "± std": 100 * full["acc_std"],
        "UCM macro-F1 present (100%)": 100 * full["f1_present"],
        "absent-class preds (100%)": 100 * full["absent_share"],
        "best UCM acc": 100 * best["acc_mean"],
    }).reindex(tr_models).round(1)
    table["at fraction"] = (100 * best["train_fraction"]).reindex(tr_models).map(lambda f: f"{f:g}%")
    table.index = [style(m)["label"] for m in table.index]
    display(table)

### 9a. Transfer Accuracy vs. Labelled EuroSAT Images

Same x-axis as the label-efficiency plot, but the y-axis is zero-shot accuracy on UC Merced. Dashed black lines mark chance and the majority-class baseline on the target. Anything below the majority line is worse than always predicting `Residential`.

In [ ]:
if not tr_summary.empty:
    panels = [("Fine-tuning", [m for m in tr_models if not m.endswith("_linear_probe") and m != "linear"]),
              ("Linear probe (frozen backbone)", [m for m in tr_models if m.endswith("_linear_probe") or m == "linear"])]
    ticks = tr_summary.drop_duplicates("train_fraction").sort_values("n_train")

    fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
    for ax, (title, panel_models) in zip(axes, panels):
        for m in panel_models:
            st, s = style(m), tr_summary[tr_summary["model"] == m].sort_values("n_train")
            if s.empty:
                continue
            ax.plot(s["n_train"], 100 * s["acc_mean"], color=st["color"],
                    ls=st["ls"], marker=st["marker"], lw=1.5, ms=5, alpha=0.9, label=st["label"])
        ax.axhline(MAJORITY, color="black", ls=(0, (4, 3)), lw=1.1, alpha=0.6)
        ax.axhline(CHANCE, color="black", ls=(0, (1, 2)), lw=1.1, alpha=0.6)
        ax.text(ticks["n_train"].iloc[-1], MAJORITY + 0.6, f"majority class ({MAJORITY:.1f}%)", ha="right", fontsize=9, alpha=0.7)
        ax.text(ticks["n_train"].iloc[-1], CHANCE + 0.6, f"chance ({CHANCE:g}%)", ha="right", fontsize=9, alpha=0.7)
        ax.set_xscale("symlog", linthresh=300)
        ax.set_xlim(-20, ticks["n_train"].iloc[-1] * 1.15)  # tight margin left of the 0-label point
        ax.xaxis.set_minor_locator(NullLocator())
        ax.set_xticks(ticks["n_train"])
        ax.set_xticklabels([f"{n}\n({100 * f:g}%)" for n, f in zip(ticks["n_train"], ticks["train_fraction"])])
        ax.set_xlabel("Labelled EuroSAT training images (fraction of train split)")
        ax.set_title(title, fontweight="bold")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, frameon=True, facecolor="white", framealpha=0.9, loc="upper left")
    axes[0].set_ylabel("Zero-shot accuracy on UC Merced (%)")
    axes[0].set_ylim(0, max(45, 100 * tr_summary["acc_mean"].max() + 3))
    fig.suptitle(f"Cross-sensor transfer EuroSAT to UC Merced, mean over seeds, lr={BASE_LR:g}", fontsize=14)
    fig.tight_layout()
    savefig(fig, "transfer_label_efficiency")
    plt.show()

### 9b. Source Accuracy vs. Transfer Accuracy

Each marker is one (model, training fraction) averaged over seeds; the line through a model's markers follows increasing training-set size. If in-domain accuracy predicted out-of-domain accuracy, the curves would rise to the upper right. Instead they go **right but not up**: more EuroSAT labels sharpen EuroSAT accuracy and leave UC Merced accuracy flat or falling.

In [ ]:
if not tr_summary.empty:
    s_all = tr_summary[tr_summary["train_fraction"] > 0]
    fig, ax = plt.subplots(figsize=(10, 7.5))
    for m in tr_models:
        st, s = style(m), s_all[s_all["model"] == m].sort_values("n_train")
        if s.empty:
            continue
        ax.plot(100 * s["source_acc"], 100 * s["acc_mean"], color=st["color"], ls=st["ls"], lw=1.0, alpha=0.5)
        ax.scatter(100 * s["source_acc"], 100 * s["acc_mean"], color=st["color"], marker=st["marker"],
                   s=18 + 70 * np.log10(s["n_train"] / s["n_train"].min() + 1), edgecolor="white", lw=0.5,
                   label=st["label"], zorder=3)
    ax.axhline(MAJORITY, color="black", ls=(0, (4, 3)), lw=1.1, alpha=0.6)
    ax.axhline(CHANCE, color="black", ls=(0, (1, 2)), lw=1.1, alpha=0.6)
    ax.text(1, MAJORITY + 0.6, f"majority class on UC Merced ({MAJORITY:.1f}%)", fontsize=9, alpha=0.7)
    ax.text(1, CHANCE + 0.6, f"chance ({CHANCE:g}%)", fontsize=9, alpha=0.7)
    ax.set(xlabel="EuroSAT test accuracy (%)", ylabel="Zero-shot accuracy on UC Merced (%)",
           xlim=(0, 102), ylim=(0, max(45, 100 * s_all["acc_mean"].max() + 3)),
           title="In-domain accuracy does not predict cross-sensor accuracy\n(marker size grows with EuroSAT training-set size)")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8.5, frameon=True, facecolor="white", framealpha=0.9, loc="upper left", ncol=2)
    fig.tight_layout()
    savefig(fig, "transfer_source_vs_target")
    plt.show()

### 9c. Which Classes Survive the Sensor Change?

Per-class recall on UC Merced for the checkpoints trained at `TRANSFER_FRACTION` of EuroSAT (mean over seeds). The last column is the share of all predictions that fell on one of the three EuroSAT classes with no UC Merced images.

In [ ]:
if not tr_summary.empty:
    TRANSFER_FRACTION = 1.0  # training fraction of the EuroSAT checkpoints to inspect

    sel = tr[tr["train_fraction"] == TRANSFER_FRACTION]
    if sel.empty:
        print(f"No transfer evals for train_fraction={TRANSFER_FRACTION}. Available: {sorted(tr['train_fraction'].unique())}")
    else:
        present_names = [c for c, p in zip(CLASS_NAMES, tr_present) if p]
        rows, labels = [], []
        for m in tr_models:
            g = sel[sel["model"] == m]
            if g.empty:
                continue
            recall = np.stack(g["recall"].tolist())[:, tr_present].mean(axis=0)
            rows.append(np.r_[recall, g["absent_pred_share"].mean()])
            labels.append(f"{style(m)['label']}  ({100 * g['acc'].mean():.0f}%)")
        heat = np.array(rows)

        fig, ax = plt.subplots(figsize=(13, 0.55 * len(rows) + 2))
        im = ax.imshow(heat, cmap="Blues", vmin=0, vmax=1, aspect="auto")
        for i in range(heat.shape[0]):
            for j in range(heat.shape[1]):
                ax.text(j, i, f"{heat[i, j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if heat[i, j] > 0.55 else "black")
        ax.set_xticks(range(heat.shape[1]))
        wrap = lambda c: c.replace("Vegetation", "\nVegetation").replace("Crop", "\nCrop")
        ax.set_xticklabels([f"{wrap(c)}\n(n={int(n)})" for c, n in zip(present_names, tr_support[tr_present])] + ["predicted an\nabsent class"],
                           fontsize=9)
        ax.axvline(len(present_names) - 0.5, color="black", lw=1)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.set_title(f"UC Merced per-class recall · checkpoints trained on {100 * TRANSFER_FRACTION:g}% of EuroSAT (accuracy in brackets)",
                     fontsize=11, fontweight="bold")
        fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        fig.tight_layout()
        savefig(fig, f"transfer_class_recall_{TRANSFER_FRACTION:g}")
        plt.show()

### 9d. Transfer Confusion Matrix

Row-normalised confusion matrix on UC Merced for one checkpoint family. Rows of absent classes are empty; their columns show where the model dumps images it cannot place.

In [ ]:
if not tr_summary.empty:
    TCM_MODEL = "vit_b16_imagenet"   # e.g. "resnet50_imagenet", "resnet50_ssl4eo_linear_probe"
    TCM_FRACTION = 0.05              # training fraction of the EuroSAT checkpoint

    if TCM_MODEL not in tr_models:
        TCM_MODEL = tr_models[0]
    sel = tr[(tr["model"] == TCM_MODEL) & (tr["train_fraction"] == TCM_FRACTION)]
    if sel.empty:
        print(f"No transfer evals for {TCM_MODEL} at fraction {TCM_FRACTION}.")
    else:
        cm = np.mean([np.array(json.loads(s)) for s in sel["transfer_confusion_json"]], axis=0)
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_norm = np.divide(cm, row_sums, out=np.full_like(cm, np.nan), where=row_sums != 0)

        fig, ax = plt.subplots(figsize=(7.5, 6.5))
        cmap = plt.cm.Blues.copy(); cmap.set_bad("#eeeeee")
        im = ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=1)
        for i in range(len(cm)):
            for j in range(len(cm)):
                if not np.isnan(cm_norm[i, j]):
                    ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=7.5,
                            color="white" if cm_norm[i, j] > 0.5 else "black")
        for j, p in enumerate(tr_present):
            if not p:
                ax.axvspan(j - 0.5, j + 0.5, color="#999999", alpha=0.12, lw=0)
        ax.set_xticks(range(len(CLASS_NAMES)))
        ax.set_xticklabels([c if p else f"{c}\n(absent)" for c, p in zip(CLASS_NAMES, tr_present)], rotation=45, ha="right", fontsize=8.5)
        ax.set_yticks(range(len(CLASS_NAMES)))
        ax.set_yticklabels([f"{c} (n={int(n)})" for c, n in zip(CLASS_NAMES, tr_support)], fontsize=8.5)
        ax.set(xlabel="Predicted", ylabel="True (UC Merced, mapped)",
               title=f"UC Merced confusion · {style(TCM_MODEL)['label']}\n{100 * TCM_FRACTION:g}% EuroSAT data · "
                     f"acc {100 * sel['acc'].mean():.1f}% · n={len(sel)} seeds")
        fig.colorbar(im, ax=ax, fraction=0.046)
        fig.tight_layout()
        savefig(fig, f"transfer_confusion_{TCM_MODEL}_{TCM_FRACTION:g}")
        plt.show()

### 9e. What the Networks Actually See

One random example per shared class from each dataset, after the pipeline's resize to the network input size. Both rows are 224 × 224 inputs; the physical footprint differs by almost a factor of ten per side. Requires both dataset caches in `data/` (`just prepare uc_merced_rgb`); the cell is skipped otherwise, never triggering a download.

In [ ]:
if not tr_summary.empty:
    from satbench.data import load_dataset

    DATA_DIR = REPO_DIR / "data"
    caches = {name: DATA_DIR / f"{name}_uint8.pt" for name in ("eurosat_rgb", TRANSFER_TARGET)}
    missing = [n for n, p in caches.items() if not p.exists()]
    if missing:
        print(f"Skipping example figure: missing dataset cache(s) {missing}. Run `just prepare <dataset>`.")
    else:
        import torch
        src_ds = load_dataset("eurosat_rgb", DATA_DIR)
        tgt_ds = load_dataset(TRANSFER_TARGET, DATA_DIR)
        rng = np.random.default_rng(0)
        shared = [i for i, p in enumerate(tr_present) if p]
        extent = {"eurosat_rgb": "≈ 640 m across (10 m/px)", TRANSFER_TARGET: "≈ 77 m across (0.3 m/px)"}

        fig, axes = plt.subplots(2, len(shared), figsize=(2.3 * len(shared), 5.4))
        for row, (name, ds) in enumerate([("eurosat_rgb", src_ds), (TRANSFER_TARGET, tgt_ds)]):
            for col, c in enumerate(shared):
                idx = np.flatnonzero(ds.labels.numpy() == c)
                img = ds.images[rng.choice(idx)]
                img = torch.nn.functional.interpolate(img[None].float(), size=(224, 224), mode="bilinear",
                                                      antialias=True, align_corners=False)[0].clamp(0, 255).byte()
                ax = axes[row, col]
                ax.imshow(img.permute(1, 2, 0).numpy())
                ax.set_xticks([]); ax.set_yticks([])
                if row == 0:
                    ax.set_title(CLASS_NAMES[c], fontsize=10)
            axes[row, 0].set_ylabel(f"{'EuroSAT' if name == 'eurosat_rgb' else 'UC Merced'}\n{extent[name]}", fontsize=9)
        fig.suptitle("Same label, different world: shared classes as the network sees them (224 × 224 inputs)", fontsize=12)
        fig.tight_layout()
        savefig(fig, "transfer_examples")
        plt.show()

### 9f. What We Learn from the Transfer Experiment

- **Zero-shot cross-sensor transfer fails for every model.** No checkpoint family beats the 33 % majority-class baseline at full data, and most stay below it, despite 94–99 % in-domain accuracy. Residential, the largest target class, is recognised by essentially none of them (recall ≈ 0): a 77 m aerial tile with three houses shares nothing but its name with a 640 m Sentinel-2 suburb. The models file those bright roofs under `Industrial` instead (see 9d), which is exactly what EuroSAT's industrial patches look like at 10 m/px.
- **In-domain accuracy is a poor proxy for robustness.** For the ResNet fine-tunes transfer accuracy is flat or declines with more EuroSAT labels while EuroSAT accuracy climbs. The models get better at the EuroSAT sensor, not at land cover. This is the classic shortcut-learning picture: the decision rule relies on scale- and sensor-specific statistics (texture frequency, colour calibration) that happen to be predictive on the source.
- **Pretraining helps transfer more than fine-tuning does.** ImageNet-pretrained linear probes transfer better than the fine-tuned versions of the same backbone (ResNet: ≈ 25 % vs ≈ 18 %), and the ImageNet ViT is the best transfer model overall (≈ 28–36 %). Frozen generic features are more scale-tolerant than features re-shaped on one sensor. Fine-tuning erodes that robustness even as it lifts in-domain accuracy.
- **SSL4EO helps the fine-tuned ResNet** (24 % vs 18 % for ImageNet), consistent with pretraining on multi-season, multi-location Sentinel-2 data, but it does not rescue the resolution gap: its probe predicts an absent class for about 15 % of images.
- **Absent-class predictions are a cheap domain-shift detector.** Up to a third of predictions land on Pasture, PermanentCrop or SeaLake. A model that routinely predicts classes that cannot occur is telling you its inputs are off-distribution.
- **The 29 % of the random-init linear probe is a false positive.** It reaches this by predicting Residential for most inputs (recall 0.69 on Residential, ≈ 0 on the vegetation classes), so it is the majority baseline in disguise. Accuracy alone is misleading on a class-imbalanced target; read it together with present-class macro-F1.
- **Caveats.** Two seeds per cell; several paired differences are within one standard deviation. The label mapping is a judgement call: UC Merced `chaparral` (desert scrub on bare soil, see 9e) becomes `HerbaceousVegetation`, `storagetanks` becomes `Industrial`, so some "errors" are disagreements about the mapping rather than about the image, and the 30× ground-sampling gap confounds sensor shift with scale shift. Turning this into a proper domain-adaptation study would need, at minimum, a few-shot adaptation baseline on UC Merced (train the head on 10–50 target images per class), resolution-matched inputs (downsample UC Merced to ~10 m/px tiles before resizing), and a label mapping validated against the EuroSAT class definitions.

## 10. Few-Shot Adaptation on UC Merced: Does EuroSAT Training Help or Hurt?

Section 9 showed that EuroSAT checkpoints transfer badly *zero-shot*. The fairer question is what happens once a few UC Merced labels are available. `configs/ucm_fewshot_linear_probe.yaml` trains **linear probes (frozen backbone)** on stratified UC Merced subsets, starting from

- the **unmodified** pretrained backbone (ImageNet-1k or SSL4EO-S12), `init_from: null`, and
- the **same backbone after fine-tuning on 1 %, 10 % or 100 % of EuroSAT** (`init_from: "@<fraction>"`, resolved to the checkpoint of the fine-tuning run with the same seed; the classification head is re-initialised).

Because only the head is trained, the curves measure the *quality of the frozen features for UC Merced*. If a EuroSAT-adapted curve lies above the unmodified one, EuroSAT taught the backbone something that survives the 30× resolution change. If it lies below, EuroSAT fine-tuning overwrote generic features with sensor-specific ones. Fractions translate into shots per class (100-image classes / Highway / Residential): 2 % → 1 / 4 / 5, 5 % → 3 / 9 / 12, 10 % → 6 / 18 / 24, 25 % → 15 / 45 / 60, 50 % → 30 / 90 / 120, 100 % → 60 / 180 / 240. The dotted horizontal line is the best zero-shot transfer accuracy of that backbone family from Section 9.

In [ ]:
FEWSHOT_EXPERIMENT = "ucm_fewshot_linear_probe"
FEWSHOT_EPOCHS = 25   # epoch budget to show (None = newest available)

fs = runs[(runs["experiment"] == FEWSHOT_EXPERIMENT) & (runs["status"] == "completed")].copy()
if not INCLUDE_DEBUG:
    fs = fs[fs["eval_limit"].isna()]
if "init_from" not in fs.columns:
    fs["init_from"] = np.nan

if fs.empty:
    print(f"No runs for experiment '{FEWSHOT_EXPERIMENT}' yet. Run: "
          f"uv run satbench run --config configs/{FEWSHOT_EXPERIMENT}.yaml --user <you> --seeds 0")
    fs_summary = pd.DataFrame()
else:
    if FEWSHOT_EPOCHS is not None and (fs["epochs"] == FEWSHOT_EPOCHS).any():
        fs = fs[fs["epochs"] == FEWSHOT_EPOCHS]
    fs["pipeline_version"] = pd.to_numeric(fs["pipeline_version"])
    fs = fs[fs["pipeline_version"] == fs.groupby("model")["pipeline_version"].transform("max")]

    # Describe the backbone init: the source run's model and EuroSAT fraction, looked up by config_hash
    src_meta = runs.drop_duplicates("config_hash").set_index("config_hash")[["model", "train_fraction", "dataset"]]
    fs["family"] = fs["model"].str.removesuffix("_linear_probe")
    fs["src_fraction"] = fs["init_from"].map(src_meta["train_fraction"])          # NaN for unmodified
    fs["src_model"] = fs["init_from"].map(src_meta["model"])
    fs["init_label"] = np.where(fs["init_from"].isna(), "unmodified",
                                "EuroSAT " + (100 * fs["src_fraction"].fillna(0)).map(lambda f: f"{f:g}") + "% fine-tune")
    unresolved = fs["init_from"].notna() & fs["src_fraction"].isna()
    if unresolved.any():
        print(f"Warning: {unresolved.sum()} runs reference source hashes not found in results/runs; dropped.")
        fs = fs[~unresolved]

    fs_summary = (fs.groupby(["family", "init_label", "src_fraction", "train_fraction"], dropna=False, as_index=False)
                    .agg(n_train=("n_train", "first"), n_seeds=("seed", "nunique"),
                         acc_mean=("test_acc", "mean"), acc_std=("test_acc", "std"),
                         f1_mean=("test_macro_f1", "mean")))
    fs_summary["acc_std"] = fs_summary["acc_std"].fillna(0.0)

    print(f"{len(fs)} runs | families: {sorted(fs['family'].unique())} | inits: {sorted(fs['init_label'].unique())} | "
          f"seeds: {sorted(int(s) for s in fs['seed'].unique())} | lr={fs['lr'].iloc[0]:g} | epochs={sorted(fs['epochs'].unique())}")
    display((fs_summary.assign(acc=lambda d: (100 * d["acc_mean"]).round(1))
                       .pivot_table(index=["family", "init_label"], columns="train_fraction", values="acc", sort=False)))

### 10a. Probe Accuracy vs. Labelled UC Merced Images

One panel per backbone family. Solid: the unmodified pretrained backbone. Dashed/dotted: the same backbone after EuroSAT fine-tuning, darker for more EuroSAT data. Bands are ± 1 std over seeds where more than one seed exists.

In [ ]:
if not fs_summary.empty:
    families = [f for f in ("resnet50_imagenet", "resnet50_ssl4eo", "vit_b16_imagenet") if f in set(fs_summary["family"])]
    families += sorted(set(fs_summary["family"]) - set(families))
    SRC_LS = {0.01: (0, (1, 1.5)), 0.1: (0, (4, 2)), 1.0: (0, (6, 2, 1, 2))}   # line style per EuroSAT fraction
    SRC_MARK = {0.01: "^", 0.1: "s", 1.0: "D"}

    ticks = fs_summary.drop_duplicates("train_fraction").sort_values("n_train")
    n_test = int(fs["n_test"].iloc[0])
    # UC Merced test split is stratified, so the majority-class rate equals the dataset's (Residential 400/1200)
    fs_majority = 100 / 3
    fs_chance = 100 / 10

    fig, axes = plt.subplots(1, len(families), figsize=(7.5 * len(families), 7), sharey=True, squeeze=False)
    for ax, fam in zip(axes[0], families):
        base_color = style(fam)["color"]
        d = fs_summary[fs_summary["family"] == fam]
        # unmodified backbone
        s = d[d["init_label"] == "unmodified"].sort_values("n_train")
        if not s.empty:
            ax.plot(s["n_train"], 100 * s["acc_mean"], color=base_color, ls="-", marker="o", lw=2.4, ms=7,
                    label=f"{style(fam)['label'].split(' (')[0]}, unmodified")
            if s["n_seeds"].max() > 1:
                ax.fill_between(s["n_train"], 100 * (s["acc_mean"] - s["acc_std"]), 100 * (s["acc_mean"] + s["acc_std"]),
                                color=base_color, alpha=0.1, lw=0)
        # EuroSAT-adapted backbones
        for frac, g in d[d["init_label"] != "unmodified"].groupby("src_fraction"):
            g = g.sort_values("n_train")
            shade = 0.45 + 0.55 * np.log10(frac * 100 + 1) / np.log10(101)
            ax.plot(g["n_train"], 100 * g["acc_mean"], color=base_color, alpha=shade, ls=SRC_LS.get(frac, "--"),
                    marker=SRC_MARK.get(frac, "x"), lw=1.6, ms=5.5, label=f"after EuroSAT fine-tune on {100 * frac:g}%")
            if g["n_seeds"].max() > 1:
                ax.fill_between(g["n_train"], 100 * (g["acc_mean"] - g["acc_std"]), 100 * (g["acc_mean"] + g["acc_std"]),
                                color=base_color, alpha=0.06, lw=0)
        # best zero-shot transfer of this family (Section 9), if available
        if "tr_summary" in globals() and not tr_summary.empty:
            z = tr_summary[tr_summary["model"].str.removesuffix("_linear_probe") == fam]
            if not z.empty:
                zbest = 100 * z["acc_mean"].max()
                ax.axhline(zbest, color=base_color, ls=(0, (1, 3)), lw=1.3, alpha=0.8)
                y_off = 3.2 if abs(zbest - fs_majority) < 3 else 0.7
                ax.text(ticks["n_train"].iloc[-1], zbest + y_off, f"best zero-shot transfer ({zbest:.0f}%)",
                        ha="right", fontsize=9, color=base_color, alpha=0.9)
        ax.axhline(fs_majority, color="black", ls=(0, (4, 3)), lw=1.0, alpha=0.5)
        ax.axhline(fs_chance, color="black", ls=(0, (1, 2)), lw=1.0, alpha=0.5)
        ax.text(ticks["n_train"].iloc[-1], fs_majority + 0.7, f"majority class ({fs_majority:.1f}%)", ha="right", fontsize=9, alpha=0.6)
        ax.text(ticks["n_train"].iloc[-1], fs_chance + 0.7, "chance (10%)", ha="right", fontsize=9, alpha=0.6)
        ax.set_xscale("symlog", linthresh=30)
        ax.set_xlim(-2, ticks["n_train"].iloc[-1] * 1.15)  # tight margin left of the 0-label point
        ax.xaxis.set_minor_locator(NullLocator())
        ax.set_xticks(ticks["n_train"])
        ax.set_xticklabels([f"{n}\n({100 * f:g}%)" for n, f in zip(ticks["n_train"], ticks["train_fraction"])])
        ax.set_xlabel("Labelled UC Merced training images (fraction of train split)")
        ax.set_title(f"{style(fam)['label'].split(' (')[0]} backbone, linear probe", fontweight="bold")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, frameon=True, facecolor="white", framealpha=0.9, loc="lower right")
    axes[0, 0].set_ylabel(f"UC Merced test accuracy (%)  [n={n_test}]")
    axes[0, 0].set_ylim(0, 100)
    fig.suptitle("Few-shot adaptation on UC Merced: unmodified pretrained backbone vs. after EuroSAT fine-tuning",
                 fontsize=14)
    fig.tight_layout()
    savefig(fig, "fewshot_ucm_linear_probe")
    plt.show()

### 10b. Gain from EuroSAT Fine-Tuning per Shot Budget

Difference in UC Merced probe accuracy between the EuroSAT-adapted backbone and the unmodified one, at every labelled-data budget. Positive = EuroSAT fine-tuning produced better frozen features for UC Merced.

In [ ]:
if not fs_summary.empty:
    base = (fs_summary[fs_summary["init_label"] == "unmodified"]
              .set_index(["family", "train_fraction"])["acc_mean"])
    # the 0 % point is an untrained head and carries no information about the features
    adapted = fs_summary[(fs_summary["init_label"] != "unmodified") & (fs_summary["train_fraction"] > 0)].copy()
    ticks = ticks[ticks["train_fraction"] > 0]
    adapted["gain_pp"] = 100 * (adapted["acc_mean"].values
                                - base.reindex(list(zip(adapted["family"], adapted["train_fraction"]))).values)
    gain = adapted.pivot_table(index=["family", "init_label"], columns="train_fraction", values="gain_pp", sort=False).round(1)
    display(gain)

    fig, axes = plt.subplots(1, len(families), figsize=(7 * len(families), 4.8), sharey=True, squeeze=False)
    for ax, fam in zip(axes[0], families):
        g_all = adapted[adapted["family"] == fam]
        fracs = sorted(g_all["src_fraction"].unique())
        width = 0.8 / max(len(fracs), 1)
        xs = np.arange(len(ticks))
        for k, frac in enumerate(fracs):
            g = g_all[g_all["src_fraction"] == frac].set_index("train_fraction").reindex(ticks["train_fraction"])
            shade = 0.45 + 0.55 * np.log10(frac * 100 + 1) / np.log10(101)
            ax.bar(xs + (k - (len(fracs) - 1) / 2) * width, g["gain_pp"], width, color=style(fam)["color"], alpha=shade,
                   label=f"after EuroSAT {100 * frac:g}%")
        ax.axhline(0, color="black", lw=1)
        ax.set_xticks(xs)
        ax.set_xticklabels([f"{n}\n({100 * f:g}%)" for n, f in zip(ticks["n_train"], ticks["train_fraction"])], fontsize=9)
        ax.set_xlabel("Labelled UC Merced training images")
        ax.set_title(f"{style(fam)['label'].split(' (')[0]}: gain over unmodified backbone", fontweight="bold")
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend(fontsize=9, frameon=True)
    axes[0, 0].set_ylabel("Accuracy gain (percentage points)")
    fig.tight_layout()
    savefig(fig, "fewshot_ucm_gain")
    plt.show()

### 10c. Reading the Few-Shot Results

Seed 0 only so far (seeds 1 and 2 need the EuroSAT checkpoints of that seed on the machine that runs them; `uv run satbench run --config configs/ucm_fewshot_linear_probe.yaml --user <you> --seeds 1`).

- **A handful of target labels fixes most of the zero-shot failure.** With one UC Merced image per class (14 images) the unmodified ImageNet probe reaches 77 %, with three per class 85 %, against a best zero-shot transfer of 36 %. The features were never the problem; the label mapping and the 30× scale change were.
- **EuroSAT fine-tuning makes the frozen backbone worse for UC Merced at every labelled budget.** For ImageNet the adapted backbones trail the unmodified one by 13–21 points at one shot per class and 4–10 points at six shots. For SSL4EO the gap is 10–18 points through most of the curve.
- **The two backbones recover differently.** With all 720 UC Merced training images the ImageNet gap closes to 1–3 points: the EuroSAT-shifted features remain almost as linearly separable, they just need more target labels to find the new boundary. The SSL4EO gap does not close (82–90 % vs 97 %), and it grows with the amount of EuroSAT data used (1 %: −7, 10 %: −9, 100 %: −15 points at full data). A backbone that already encodes Sentinel-2 statistics is pushed further into that sensor by EuroSAT fine-tuning, and the aerial imagery falls outside what its features still separate.
- **Little EuroSAT data is enough to do the damage** for ImageNet: 1 % of EuroSAT (162 images, 25 epochs at lr 1e-4) costs as much at low shot budgets as 100 % does. This points at early, cheap changes during fine-tuning rather than slowly learned EuroSAT-specific filters. The obvious suspect is BatchNorm: fine-tuning overwrites the running statistics with Sentinel-2 statistics, and a frozen probe then normalises brighter, higher-contrast aerial images with the wrong mean and variance. Re-estimating only the BatchNorm statistics on unlabelled UC Merced images (no labels, no gradient steps) and re-probing would tell whether the gap is normalisation or learned filters.
- **Practical reading.** For cross-sensor use, keep a generic pretrained backbone and spend the label budget on a few target-domain examples per class; fine-tuning on a large source dataset from another sensor does not buy robustness and can cost it.
- **Caveats.** One seed, 240 test images (so one image is 0.4 points), and probe-only by design: a full fine-tune on UC Merced would very likely erase these differences, as EuroSAT fine-tuning erased the ImageNet-vs-SSL4EO gap in Section 4.